<a href="https://colab.research.google.com/github/VarshaNeelvani/InnomaticsInternship/blob/main/BERT_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ASSIGNMENT NLP – 4 (BERT Fine-Tuning)**


## **Assignment: Fine-Tuning BERT on a Kaggle Dataset**

### Install Libraries

In [1]:
!pip install transformers datasets scikit-learn

###Import Libraries

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

###Load Dataset

In [3]:
from google.colab import files
uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [4]:
df = pd.read_csv("IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


###Data Preprocessing

In [5]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)  # remove HTML
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['review'] = df['review'].apply(clean_text)

In [6]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [7]:
df.dropna(inplace=True)

###Train-Test Split

In [8]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['sentiment'], test_size=0.3, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

###Tokenization (BERT)

In [9]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

###Create Dataset Class

In [10]:
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

##DataLoader

In [11]:
train_dataset = CustomDataset(train_encodings, train_labels)
val_dataset = CustomDataset(val_encodings, val_labels)
test_dataset = CustomDataset(test_encodings, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

###Load BERT Model

In [12]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


###Optimizer

In [13]:
optimizer = AdamW(model.parameters(), lr=2e-5)

###Training Loop

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(2):
    model.train()

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} completed")

Epoch 1 completed
Epoch 2 completed


###Evaluation Metrics

In [15]:
def evaluate(model, loader):
    model.eval()

    preds, true = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
            true.extend(batch['labels'].numpy())

    acc = accuracy_score(true, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(true, preds, average='binary')
    cm = confusion_matrix(true, preds)

    return acc, precision, recall, f1, cm

###Test Results

In [16]:
test_loader = DataLoader(test_dataset, batch_size=16)
acc, precision, recall, f1, cm = evaluate(model, test_loader)

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", cm)

Accuracy: 0.8957333333333334
Precision: 0.8937960042060988
Recall: 0.899947061937533
F1 Score: 0.8968609865470852
Confusion Matrix:
 [[3318  404]
 [ 378 3400]]


###Experiment 1: Freeze BERT

In [17]:
def train_model(model, train_loader, epochs=2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")

In [19]:
train_model(model, train_loader)

acc1, p1, r1, f1_1, cm1 = evaluate(model, test_loader)

Epoch 1, Avg Loss: 0.0685
Epoch 2, Avg Loss: 0.0377


###Experiment 2: Fine-tune last 2 layers

In [20]:
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
        print(f"Trainable: {name}")
    else:
        param.requires_grad = False

Trainable: encoder.layer.10.attention.self.query.weight
Trainable: encoder.layer.10.attention.self.query.bias
Trainable: encoder.layer.10.attention.self.key.weight
Trainable: encoder.layer.10.attention.self.key.bias
Trainable: encoder.layer.10.attention.self.value.weight
Trainable: encoder.layer.10.attention.self.value.bias
Trainable: encoder.layer.10.attention.output.dense.weight
Trainable: encoder.layer.10.attention.output.dense.bias
Trainable: encoder.layer.10.attention.output.LayerNorm.weight
Trainable: encoder.layer.10.attention.output.LayerNorm.bias
Trainable: encoder.layer.10.intermediate.dense.weight
Trainable: encoder.layer.10.intermediate.dense.bias
Trainable: encoder.layer.10.output.dense.weight
Trainable: encoder.layer.10.output.dense.bias
Trainable: encoder.layer.10.output.LayerNorm.weight
Trainable: encoder.layer.10.output.LayerNorm.bias
Trainable: encoder.layer.11.attention.self.query.weight
Trainable: encoder.layer.11.attention.self.query.bias
Trainable: encoder.layer.1